# IKONIC: Agentic AI & RAG Practice Notebook

This notebook contains practical exercises for building **Agentic AI** systems using **LangChain** and **LangGraph**, as required for the IKONIC Generative AI Engineer role.

## Setup
You will need an OpenAI or Anthropic API key to run these examples.

In [ ]:
!pip install -qU langchain langchain-openai langgraph tavily-python

## 1. Simple ReAct Agent
A ReAct agent uses a "Thought-Action-Observation" loop.

In [ ]:
import os
from google.colab import userdata # If using Colab

os.environ["OPENAI_API_KEY"] = "your_api_key"
os.environ["TAVILY_API_KEY"] = "your_tavily_key" # For web search

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import create_react_agent

model = ChatOpenAI(model="gpt-4o")
tools = [TavilySearchResults(max_results=2)]

# Create the agent
app = create_react_agent(model, tools)

# Invoke the agent
response = app.invoke({"messages": [("user", "What is the current weather in Islamabad?")]})
print(response["messages"][-1].content)

## 2. LangGraph: Cyclic Workflows
Building a custom graph where an agent writes code and a second agent reviews it.

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, END

class State(TypedDict):
    code: str
    review: str
    iterations: int

def coder(state: State):
    print("---CODER WORKING---")
    # In a real scenario, this would be an LLM call
    return {"code": "print('hello world')", "iterations": state.get("iterations", 0) + 1}

def reviewer(state: State):
    print("---REVIEWER CHECKING---")
    if "hello" in state["code"]:
        return {"review": "Approved"}
    return {"review": "Rejected"}

def should_continue(state: State):
    if state["review"] == "Approved" or state["iterations"] > 3:
        return END
    return "coder"

workflow = StateGraph(State)
workflow.add_node("coder", coder)
workflow.add_node("reviewer", reviewer)

workflow.set_entry_point("coder")
workflow.add_edge("coder", "reviewer")
workflow.add_conditional_edges("reviewer", should_continue)

app = workflow.compile()
app.invoke({"code": "", "review": "", "iterations": 0})

## 3. RAG with Vector DB
Practicing document ingestion and retrieval.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = "IKONIC is a US-based IT company headquartered in Miami, Florida. They specialize in Agentic AI."

splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
chunks = splitter.create_documents([text])

vector_db = FAISS.from_documents(chunks, OpenAIEmbeddings())
retriever = vector_db.as_retriever()

docs = retriever.invoke("Where is IKONIC headquartered?")
print(docs[0].page_content)

## Practice Exercises
1. **Multi-Agent:** Add a third agent "Documenter" to the graph above.
2. **Tool Use:** Create a tool that calculates the square root and give it to a ReAct agent.
3. **RAG Evaluation:** Try retrieval with different `chunk_size` and see how it affects the results.